# Phase 2 — first end-to-end run (smoke test)
Run the cells **one by one** (Shift+Enter). Goal: prove the pipeline works (GPU → BasicTS → STID on PEMS04 for 2 epochs → one results JSON).
These numbers are a **pipeline check, not a result**: status is saved as `smoke_test`.
Before starting: *Runtime → Change runtime type → T4 GPU*.

## 1. Check GPU

In [ ]:
!nvidia-smi
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 2. Mount Google Drive (a pop-up will ask for permission)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone your repo (with the BasicTS submodule)

In [ ]:
%cd /content
!rm -rf proposal
!git clone --recursive https://github.com/hrfpour/proposal.git
%cd /content/proposal
!git log --oneline -3
!ls code/BasicTS | head -30

## 4. Install BasicTS requirements (if it asks to restart the runtime, do it, then continue from cell 5)

In [ ]:
%cd /content/proposal/code/BasicTS
!pip install -r requirements.txt 2>&1 | tail -n 25

## 5. Show the quick-start lines of YOUR BasicTS version (so we use the right commands)

In [ ]:
%cd /content/proposal/code/BasicTS
!grep -n -i -E "python experiments|unzip|all_data|generate_training_data|datasets/" README.md | head -40

## 6. Unpack the data you copied to your Drive (edit ZIP_PATH if needed)

In [ ]:
import os
ZIP_PATH = "/content/drive/MyDrive/all_data.zip"
assert os.path.exists(ZIP_PATH), f"Not found: {ZIP_PATH}"
%cd /content/proposal/code/BasicTS
!mkdir -p datasets
!unzip -q -o "{ZIP_PATH}" -d datasets/
!ls datasets
!ls datasets/PEMS04

## 7. Generate processed data only if it is missing

In [ ]:
import os
if (not os.path.exists("datasets/PEMS04/desc.json")) and os.path.exists("scripts/data_preparation/PEMS04/generate_training_data.py"):
    !python scripts/data_preparation/PEMS04/generate_training_data.py
else:
    print("Skip: processed data already present, or no generator script in this version.")
!ls datasets/PEMS04

## 8. Find the STID config for PEMS04 and make a 2-epoch copy

In [ ]:
import glob, re, pathlib
cfgs = sorted(glob.glob("baselines/STID/*PEMS04*.py"))
print("configs found:", cfgs)
src = pathlib.Path(cfgs[0])
dst = src.with_name("PEMS04_smoke.py")
txt = src.read_text()
new, n = re.subn(r"(NUM_EPOCHS\s*=\s*)\d+", r"\g<1>2", txt)
print("NUM_EPOCHS replaced:", n, "(should be 1)")
dst.write_text(new)
print("smoke config:", dst)

## 9. Train for 2 epochs

In [ ]:
!python experiments/train.py -c {dst} --gpus '0' 2>&1 | tee /content/smoke_run.log | tail -n 60

## 10. Export the results JSON and copy it to Drive

In [ ]:
!python /content/proposal/code/scripts/export_result.py --log-dir checkpoints --model STID --dataset PEMS04 --seed 0 --config {dst} --epochs 2 --status smoke_test --out /content/results_out
!ls /content/results_out
import shutil, time
dest = f"/content/drive/MyDrive/proposal_runs/{time.strftime('%Y%m%d_%H%M')}"
shutil.copytree("/content/results_out", dest)
print("Saved to Drive:", dest)